# 02. 공시 노출 라벨링 준비 (파일럿 P2)

S1 상위 5개 발행사의 10-K를 문단 단위로 추출하고, 사람이 정답 라벨을 붙일 시트를 만든다.

- 판정 기준: `labeling_guide.md` v1.1 (라벨링 전에 커밋되어 있어야 한다)
- 이 노트북은 **판정을 하지 않는다.** 키워드 적중 목록은 읽기 보조 도구다.
- LLM 추출은 이 노트북에 없다. 라벨 시트를 커밋한 **뒤에** 별도 노트북에서 실행한다.

**선행 조건:** `01_kic_13f_extraction.ipynb`를 실행해 `data/processed/S1/sample.csv`가 있어야 한다.

In [1]:
# ── 설정 ─────────────────────────────────────────────
USE_FIXTURES = False   # True: 모의 응답 (01 노트북도 모의 모드로 먼저 실행해야 함)
USER_AGENT = "Blind Spot pilot Hyeseong Kim luckykhs@gachon.ac.kr"   # ← 01 노트북과 같게

TOP_N = 5              # 프로토콜 7.1: 상위 5개 발행사
LABELER = "Hyeseong Kim"   # 라벨 작성자 (labels_P2.csv 의 labeler 열)

In [2]:
import sys, textwrap
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "tests")]
import kic13f as k
import filings as fl

if USE_FIXTURES:
    from fixtures import FakeClient
    client = FakeClient()
    S1_PATH = ROOT / "data" / "processed_fixture" / "S1" / "sample.csv"
    LABEL_DIR = ROOT / "data" / "labeling_fixture"
    print("⚠ 모의 응답 모드: 실제 공시가 아니다.")
else:
    client = k.SecClient(USER_AGENT, cache_dir=ROOT / "data" / "raw" / "sec_cache")
    S1_PATH = ROOT / "data" / "processed" / "S1" / "sample.csv"
    LABEL_DIR = ROOT / "data" / "labeling"

DOC_DIR = LABEL_DIR / "docs"
HIT_DIR = LABEL_DIR / "hits"
LABEL_PATH = LABEL_DIR / "labels_P2.csv"
for d in (DOC_DIR, HIT_DIR):
    d.mkdir(parents=True, exist_ok=True)

sample = pd.read_csv(S1_PATH).sort_values("rank")
sample[["rank", "cik", "edgar_name", "latest_10k_before_cutoff", "latest_10k_filing_date"]].head(TOP_N)

,rank,cik,edgar_name,latest_10k_before_cutoff,latest_10k_filing_date
0,1,1045810,NVIDIA CORP,0001045810-26-000021,2026-02-25
1,2,320193,Apple Inc.,0000320193-25-000079,2025-10-31
2,3,1652044,Alphabet Inc.,0001652044-26-000018,2026-02-05
3,4,789019,MICROSOFT CORP,0001193125-26-323660,2026-07-29
4,5,1018724,AMAZON COM INC,0001018724-26-000004,2026-02-06


## 1. 10-K 원문 추출

각 발행사의 `latest_10k_before_cutoff` 10-K 본문을 문단으로 나누고, 문단마다 ID(`항목-p순번`)를 붙인다.
인용할 때 이 ID를 `evidence_location`에 적는다.

In [3]:
docs = {}

def extract(row):
    url = fl.primary_document_url(client, row.cik, row.latest_10k_before_cutoff)
    df = fl.assign_items(fl.html_to_paragraphs(client.get_bytes(url)))
    df.to_csv(DOC_DIR / f"{row.cik}_{row.latest_10k_before_cutoff}.csv", index=False, encoding="utf-8-sig")
    docs[int(row.cik)] = df
    return url

for row in sample.head(TOP_N).itertuples(index=False):
    url = extract(row)
    print(f"[{row.rank}] {row.edgar_name:<28} 문단 {len(docs[row.cik]):>5}개  {url}")

[1] NVIDIA CORP                  문단  1845개  https://www.sec.gov/Archives/edgar/data/1045810/000104581026000021/nvda-20260125.htm
[2] Apple Inc.                   문단  1213개  https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm
[3] Alphabet Inc.                문단  2183개  https://www.sec.gov/Archives/edgar/data/1652044/000165204426000018/goog-20251231.htm
[4] MICROSOFT CORP               문단  2112개  https://www.sec.gov/Archives/edgar/data/789019/000119312526323660/msft-20260630.htm
[5] AMAZON COM INC               문단  1654개  https://www.sec.gov/Archives/edgar/data/1018724/000101872426000004/amzn-20251231.htm


### 항목 구분 점검

라벨링 범위(Item 1, 1A, 7과 재무제표 주석)가 잡혔는지 확인한다. 재무제표 주석은 회사에 따라
**Item 8 또는 Item 15**에 수록되므로(기준서 v1.1), `fin_notes_in` 열에서 실제 위치를 확인한다.
`ok = False`인 발행사는 `docs/*.csv`에서 해당 부분을 직접 찾아 읽는다(기준서 4절 2단계).

In [4]:
coverage = pd.DataFrame([{"rank": row.rank, "company": row.edgar_name, **fl.coverage_check(docs[row.cik])}
                         for row in sample.head(TOP_N).itertuples(index=False)])
display(coverage)
print("⚠ 점검 필요:", coverage.loc[~coverage["ok"], "company"].tolist() or "없음")

,rank,company,Item 1,Item 1A,Item 7,Item 8,Item 15,fin_notes_in,ok,issue
0,1,NVIDIA CORP,128,276,197,2,982,Item 15,True,
1,2,Apple Inc.,57,119,141,592,87,Item 8,True,
2,3,Alphabet Inc.,91,188,351,1153,118,Item 8,True,
3,4,MICROSOFT CORP,124,135,89,301,36,Item 8,True,
4,5,AMAZON COM INC,62,167,197,910,54,Item 8,True,


⚠ 점검 필요: 없음


## 2. 키워드 적중 목록 (읽기 보조)

테마별로 키워드가 들어 있는 문단을 모아 `hits/` 폴더에 저장한다. **적중했다고 노출이 있다는 뜻이 아니고,
적중하지 않았다고 노출이 없다는 뜻도 아니다.** 판정은 라벨링 기준서 3·4절을 따른다.

In [5]:
counts = []
for row in sample.head(TOP_N).itertuples(index=False):
    rec = {"rank": row.rank, "company": row.edgar_name}
    for theme in ("T1", "T2a", "T2b"):
        hits = fl.keyword_hits(docs[row.cik], theme)
        hits.to_csv(HIT_DIR / f"r{row.rank}_{row.cik}_{theme}.csv", index=False, encoding="utf-8-sig")
        rec[theme] = len(hits)
    counts.append(rec)
pd.DataFrame(counts)

,rank,company,T1,T2a,T2b
0,1,NVIDIA CORP,126,35,33
1,2,Apple Inc.,8,13,1
2,3,Alphabet Inc.,77,0,3
3,4,MICROSOFT CORP,60,1,2
4,5,AMAZON COM INC,31,4,1


In [6]:
def _wrap(text, width):
    # 하이픈·긴 단어에서 줄을 나누지 않는다. 화면에서 복사한 인용이 원문과 어긋나지 않게 하기 위함.
    return textwrap.fill(text, width=width, break_on_hyphens=False, break_long_words=False)

def show_hits(rank, theme, start=0, n=15, width=110):
    """적중 문단을 노트북 안에서 바로 읽는다. 예: show_hits(1, "T2b")"""
    row = sample[sample["rank"] == rank].iloc[0]
    hits = fl.keyword_hits(docs[int(row.cik)], theme)
    print(f"[{rank}] {row.edgar_name} / {theme}: 적중 {len(hits)}개 (표시 {start}~{min(start+n, len(hits))-1})\n")
    for h in hits.iloc[start:start + n].itertuples(index=False):
        print(f"── {h.pid}  [{h.terms}]")
        print(_wrap(h.text, width), "\n")

def show_paragraph(rank, pid, context=1, width=110):
    """특정 문단과 앞뒤 문단을 본다. 예: show_paragraph(1, "7-p01234")"""
    df = docs[int(sample[sample["rank"] == rank].iloc[0].cik)]
    i = df.index[df["pid"] == pid][0]
    for r in df.iloc[max(i - context, 0): i + context + 1].itertuples(index=False):
        print(f"── {r.pid}" + ("  ◀" if r.pid == pid else ""))
        print(_wrap(r.text, width), "\n")

def find_text(rank, pattern, items=None, n=20, width=110):
    """문서 전체에서 원하는 표현을 찾는다(대소문자 무관). 예: find_text(1, "Greater China|Hong Kong")"""
    df = docs[int(sample[sample["rank"] == rank].iloc[0].cik)]
    if items:
        df = df[df["item"].isin(items)]
    m = df[df["text"].str.contains(pattern, case=False, regex=True)]
    print(f"'{pattern}' 일치 {len(m)}개 (표시 최대 {n})\n")
    for r in m.head(n).itertuples(index=False):
        print(f"── {r.pid}")
        print(_wrap(r.text, width), "\n")

def raw(rank, pid):
    """문단 원문을 줄바꿈 없이 그대로 출력한다. 인용을 복사할 때 이 출력에서 복사한다."""
    df = docs[int(sample[sample["rank"] == rank].iloc[0].cik)]
    print(df.loc[df["pid"] == pid, "text"].iloc[0])

show_hits(1, "T1", n=5)

[1] NVIDIA CORP / T1: 적중 126개 (표시 0~4)

── 1-p00094  [\bAI\b; accelerated computing; data ?cent(er|re)s?]
NVIDIA pioneered accelerated computing to help solve the most challenging computational problems. NVIDIA is
now a data center scale AI infrastructure company reshaping all industries. 

── 1-p00095  [\bAI\b; \bGPUs?\b; accelerated computing; artificial intelligence]
Our technology stack includes the foundational NVIDIA CUDA development platform that runs on all NVIDIA GPUs,
as well as hundreds of domain-specific software libraries, frameworks, algorithms, software development kits,
or SDKs, and application programming interfaces, or APIs. This deep and broad software stack accelerates the
performance and facilitates the deployment of NVIDIA accelerated computing for computationally intensive
workloads such as artificial intelligence, or AI, model training and inference, data analytics, scientific
computing, robotics, and 3D graphics, with vertical-specific optimizations to address 

## 3. 라벨링 시트

`labels_P2.csv`를 만든다. **이미 있으면 덮어쓰지 않는다.**

작성 방법 (자세한 기준은 `labeling_guide.md`):
- `label`: `confirmed` / `possibly_related` / `not_found`
- `exposure_direction`: `positive` / `negative` / `mixed` / `unknown` (공시가 직접 말한 방향만)
- `evidence_text`: 원문 그대로 복사, 1~2문장
- `evidence_location`: 문단 ID (예: `7-p01234`)
- `searched`: not_found일 때 확인한 범위 (예: "적중 문단 전체 + Item 1A 정독")
- `labeler`, `labeled_at`: 작성자와 일시

Excel로 열어도 한글이 깨지지 않도록 UTF-8(BOM)으로 저장한다.

In [7]:
if LABEL_PATH.exists():
    print("기존 시트를 유지한다:", LABEL_PATH.relative_to(ROOT))
else:
    fl.label_template(sample, TOP_N).to_csv(LABEL_PATH, index=False, encoding="utf-8-sig")
    print("새 시트를 만들었다:", LABEL_PATH.relative_to(ROOT))
pd.read_csv(LABEL_PATH, encoding="utf-8-sig", keep_default_na=False)

기존 시트를 유지한다: data\labeling\labels_P2.csv


,rank,cik,company,theme,label,exposure_direction,evidence_text,evidence_location,searched,note,added_by_rule,labeler,labeled_at
0,1,1045810,NVIDIA CORP,T1,,,,,,,no,,
1,1,1045810,NVIDIA CORP,T2a,,,,,,,no,,
2,1,1045810,NVIDIA CORP,T2b,,,,,,,no,,
3,2,320193,Apple Inc.,T1,,,,,,,no,,
4,2,320193,Apple Inc.,T2a,,,,,,,no,,
5,2,320193,Apple Inc.,T2b,,,,,,,no,,
6,3,1652044,Alphabet Inc.,T1,,,,,,,no,,
7,3,1652044,Alphabet Inc.,T2a,,,,,,,no,,
8,3,1652044,Alphabet Inc.,T2b,,,,,,,no,,
9,4,789019,MICROSOFT CORP,T1,,,,,,,no,,


### 라벨 입력 도구

CSV를 Excel로 직접 고치지 않고, 아래 함수로 한 행씩 입력한다. 입력 즉시 허용 값과 인용 원문 일치를 검사한다.

```python
set_label(1, "T1", "confirmed", direction="positive", quote="원문 그대로", pid="7-p00123")
set_label(3, "T2a", "not_found", searched="적중 문단 전체 + 지역별 매출 주석 확인")
progress()   # 진행 현황
```

- 같은 행을 다시 입력하면 덮어쓴다(LLM 실행 전까지만 허용, 기준서 7절).
- `labels_P2.csv`를 Excel로 열어 둔 채 실행하면 Windows에서 저장이 실패할 수 있다. 확인할 때는 `progress()`를 쓴다.

In [8]:
from datetime import datetime

def set_label(rank, theme, label, direction="unknown", quote="", pid="", searched="", note=""):
    assert label in fl.ALLOWED["label"], f"label 은 {sorted(fl.ALLOWED['label'])} 중 하나"
    assert direction in fl.ALLOWED["exposure_direction"] - {""}, "direction 은 positive/negative/mixed/unknown"
    if label == "not_found":
        assert not quote and searched, "not_found 는 quote 없이, searched(탐색 범위)를 적는다"
    else:
        assert quote and pid, "confirmed/possibly_related 는 quote 와 pid 가 필요하다"
    L = pd.read_csv(LABEL_PATH, encoding="utf-8-sig", keep_default_na=False)
    m = (L["rank"].astype(int) == rank) & (L["theme"] == theme)
    assert m.sum() == 1, f"rank {rank} / {theme} 행이 시트에 없다"
    if quote:
        v = fl.verify_quote(quote, docs[int(L.loc[m, "cik"].iloc[0])], pid)
        if v["status"] != "ok":
            print(f"✗ 저장하지 않음 — 인용 검증 {v['status']}" + (f" (실제 위치 {v['found_in']})" if v["found_in"] else ""))
            return
    L.loc[m, ["label", "exposure_direction", "evidence_text", "evidence_location", "searched", "note",
              "labeler", "labeled_at"]] = [label, direction, quote, pid, searched, note,
                                           LABELER, f"{datetime.now():%Y-%m-%d %H:%M}"]
    L.to_csv(LABEL_PATH, index=False, encoding="utf-8-sig")
    print(f"✓ 저장: rank {rank} / {theme} → {label}\n→ 작업 셀 A 를 다시 실행하면 다음 항목이 나온다.")

def progress():
    L = pd.read_csv(LABEL_PATH, encoding="utf-8-sig", keep_default_na=False)
    view = L.pivot_table(index=["rank", "company"], columns="theme", values="label", aggfunc="first")
    display(view.replace("", "·"))
    done = int((L["label"] != "").sum())
    print(f"완료 {done}/{len(L)}  |  분포 {L.loc[L['label'] != '', 'label'].value_counts().to_dict()}")

progress()

,theme,T1,T2a,T2b
rank,company,,,
1,NVIDIA CORP,·,·,·
2,Apple Inc.,·,·,·
3,Alphabet Inc.,·,·,·
4,MICROSOFT CORP,·,·,·
5,AMAZON COM INC,·,·,·


완료 0/15  |  분포 {}


In [9]:
import re

THEME_ORDER = ["T2a", "T2b", "T1"]      # 테마별로 5개 회사를 연달아 처리 (판정 기준 일관성)

CRITERIA = {
    "T2a": """[T2a 중국 수요·매출 의존]  (labeling_guide.md 3절)
  confirmed        : 중국(또는 중국 포함 지역) 매출 금액·비중이 있거나, 중국 수요 의존을 회사 매출 요인으로 명시
  possibly_related : 중국을 시장·고객으로 언급만 하고 금액·비중·의존 진술이 없음
  not_found        : 중국 언급 없음, 또는 생산·조달(공급망) 측면으로만 언급
  주의             : 대만 단독 매출은 중국에 넣지 않음. 공시된 지역명(예: Greater China)은 note 에 그대로 적기
  방향             : 숫자만 있으면 unknown. '중국 매출이 줄었다'처럼 공시가 말할 때만 positive/negative""",
    "T2b": """[T2b 대중국 수출규제 노출]  (labeling_guide.md 3절)
  confirmed        : 회사의 '특정 제품·제품군'에 미국 수출 허가 요건·제한이 적용되었다고 명시 (제품이 특정되어야 함)
  possibly_related : '수출통제가 영향을 줄 수 있다' 같은 일반적 위험 문구만 있음
  not_found        : 수출통제 언급 없음
  방향             : 매출 감소(가능성) 언급 → negative, 방향 언급 없음 → unknown""",
    "T1": """[T1 AI 인프라 투자 노출]  (labeling_guide.md 3절)
  confirmed        : '이 회사의' 제품·매출·비용·투자·수요 중 하나가 AI·데이터센터와 같은 문장(또는 바로 다음 문장)에서 연결
  possibly_related : AI를 산업 추세·전망·경쟁으로만 언급 / 규제·윤리 위험으로만 언급 / 계획만 있음 / 데이터센터만 있고 AI 연결 없음
  not_found        : AI·데이터센터 관련 언급 없음
  주의             : 'AI'가 많이 나온다고 confirmed 아님
  방향             : 매출·수요 증가 → positive, 비용 증가만 → negative, 둘 다 → mixed, 진술 없음 → unknown""",
}

AI = r"(?-i:\bAI\b)|artificial intelligence|data ?cent(?:er|re)|accelerated computing"
SEARCH_PLAN = {      # (설명, 패턴, 항목) — 위에서부터 순서대로 후보를 모은다
    "T2a": [("지역별 매출 표·주석", r"China|Hong Kong|\bPRC\b", ["8", "15"]),
            ("지역 구분 표현", r"geographic|by region|Americas|Asia.Pacific|International|Other countries", ["8", "15"]),
            ("사업·위험·경영 분석의 중국 언급", r"China|Chinese|Hong Kong", ["1", "1A", "7"])],
    "T2b": [("수출통제 직접 표현", r"export control|export licen[cs]|licen[cs]e requirement|Entity List|Export Administration", ["1", "1A", "7", "8", "15"]),
            ("위험요인의 수출 언급", r"\bexport", ["1A"])],
    "T1":  [("경영진 분석의 AI·데이터센터", AI, ["7"]),
            ("경영진 분석의 설비투자", r"capital expenditure|property and equipment", ["7"]),
            ("사업 설명의 AI·데이터센터", AI, ["1"]),
            ("위험요인·주석의 AI·데이터센터", AI, ["1A", "8", "15"])],
}

_TASK = {}

def _labels():
    return pd.read_csv(LABEL_PATH, encoding="utf-8-sig", keep_default_na=False)

def _next_pending():
    L = _labels()
    order = {t: i for i, t in enumerate(THEME_ORDER)}
    todo = L[L["label"] == ""].copy()
    if todo.empty:
        return None
    todo["_o"] = todo["theme"].map(order)
    r = todo.sort_values(["_o", "rank"]).iloc[0]
    return int(r["rank"]), r["theme"]

def label_task(rank, theme, per_page=8):
    """특정 항목의 후보 문단을 모아 보여준다. 이미 라벨한 항목을 다시 볼 때도 쓴다."""
    row = sample[sample["rank"] == rank].iloc[0]
    df = docs[int(row.cik)]
    cands, seen = [], set()
    for desc, pat, items in SEARCH_PLAN[theme]:
        rx = re.compile(pat, re.IGNORECASE)
        sub = df[df["item"].isin(items)]
        for r in sub.itertuples(index=False):
            if r.pid not in seen and rx.search(r.text):
                seen.add(r.pid)
                cands.append({"pid": r.pid, "item": r.item, "stage": desc, "text": r.text})
    _TASK.update(rank=rank, theme=theme, cands=cands, page=0, per_page=per_page, company=row.edgar_name)
    L = _labels(); done = int((L["label"] != "").sum())
    print("=" * 100)
    print(f"작업: rank {rank} {row.edgar_name} / {theme}      (전체 진행 {done}/{len(L)})")
    print("=" * 100)
    print(CRITERIA[theme], "\n")
    print(f"후보 문단 {len(cands)}개 (탐색 순서: " + " → ".join(d for d, _, _ in SEARCH_PLAN[theme]) + ")\n")
    _show_page()

def _show_page():
    t = _TASK; s = t["page"] * t["per_page"]; e = min(s + t["per_page"], len(t["cands"]))
    if not t["cands"]:
        print("후보 없음.")
    for i in range(s, e):
        cd = t["cands"][i]
        print(f"[{i}] {cd['pid']}  ({cd['stage']})")
        print(_wrap(cd["text"], 110), "\n")
    if e < len(t["cands"]):
        print(f"… {s}~{e-1}번 표시. 다음 후보: more()")
    else:
        print(f"── 후보 {len(t['cands'])}개를 모두 표시했다.")
    _print_templates()

def more():
    """다음 후보 페이지."""
    if (_TASK["page"] + 1) * _TASK["per_page"] < len(_TASK["cands"]):
        _TASK["page"] += 1
    _show_page()

def pick(i):
    """후보 번호 i 의 원문 전체를 줄바꿈 없이 보여주고, 입력 줄을 만들어 준다. 인용은 여기서 복사한다."""
    cd = _TASK["cands"][i]
    print(f"[{i}] {cd['pid']}  ({cd['stage']}) — 원문 전체:\n")
    print(cd["text"], "\n")
    show_paragraph(_TASK["rank"], cd["pid"], context=1)
    r, t = _TASK["rank"], _TASK["theme"]
    print("▼ 작업 셀 B 에 붙여넣고, label·direction·quote 를 고쳐 실행 (quote 는 위 원문에서 1~2문장 복사)")
    print(f'set_label({r}, "{t}", "confirmed", direction="unknown", quote="", pid="{cd["pid"]}", note="")')

def _print_templates():
    r, t, n = _TASK["rank"], _TASK["theme"], len(_TASK["cands"])
    print("\n▼ 판정 방법: 인용할 후보가 있으면 pick(번호) 실행 → 나온 줄을 작업 셀 B 에서 고쳐 실행")
    print("  후보를 모두 읽었는데 해당 문장이 없을 때만 아래 줄을 그대로 작업 셀 B 에 붙여 실행:")
    print(f'set_label({r}, "{t}", "not_found", searched="자동 탐색 후보 {n}개 전체 검토 (' +
          ", ".join(d for d, _, _ in SEARCH_PLAN[t]) + ')")')

def label_next():
    """아직 라벨하지 않은 다음 항목을 연다."""
    nxt = _next_pending()
    if nxt is None:
        progress()
        print("\n모든 행 라벨 완료. 아래 '4. 라벨 검증' 셀부터 실행한다.")
        return
    label_task(*nxt)

## 라벨링 작업 (15건 반복)

아래 **작업 셀 A**와 **작업 셀 B** 두 개만 번갈아 실행한다.

1. **작업 셀 A** 실행 → 다음 항목, 판정 기준, 후보 문단(번호 붙음), 입력할 줄이 나온다.
2. 후보를 읽는다. 더 보려면 셀 A 아래에 `more()`를 입력해 실행.
3. 인용할 후보가 있으면 `pick(번호)` 실행 → 원문 전체와 `set_label(...)` 줄이 나온다.
4. 그 줄을 **작업 셀 B**에 붙여넣고 `label`, `direction`, `quote`를 고쳐 실행한다.
   - `quote`에는 `pick()` 화면의 원문에서 1~2문장을 복사해 넣는다.
5. `✓ 저장`이 나오면 1번으로 돌아간다. `✗`가 나오면 메시지대로 고친다.
6. 후보를 모두 읽었는데 해당 문장이 없으면, 셀 A 맨 아래 `not_found` 줄을 셀 B에 붙여 실행한다.

판정(어떤 라벨인지)은 사람이 한다. 코드는 후보를 찾아 보여주고, 입력값과 인용을 검사만 한다.

In [10]:
label_next()   # 작업 셀 A — 실행하면 다음 항목이 열린다

작업: rank 1 NVIDIA CORP / T2a      (전체 진행 0/15)
[T2a 중국 수요·매출 의존]  (labeling_guide.md 3절)
  confirmed        : 중국(또는 중국 포함 지역) 매출 금액·비중이 있거나, 중국 수요 의존을 회사 매출 요인으로 명시
  possibly_related : 중국을 시장·고객으로 언급만 하고 금액·비중·의존 진술이 없음
  not_found        : 중국 언급 없음, 또는 생산·조달(공급망) 측면으로만 언급
  주의             : 대만 단독 매출은 중국에 넣지 않음. 공시된 지역명(예: Greater China)은 note 에 그대로 적기
  방향             : 숫자만 있으면 unknown. '중국 매출이 줄었다'처럼 공시가 말할 때만 positive/negative 

후보 문단 39개 (탐색 순서: 지역별 매출 표·주석 → 지역 구분 표현 → 사업·위험·경영 분석의 중국 언급)

[0] 15-p01596  (지역별 매출 표·주석)
We are subject to examination by taxing authorities both in the United States and other countries. As of
January 25, 2026, the significant tax jurisdictions that may be subject to examination include the United
States for fiscal years after 2022, as well as Canada, China, Germany, Hong Kong, India, Israel, Italy, and
Taiwan for fiscal years 2014 through 2025. As of January 25, 2026, the significant tax jurisdictions for which
we are currently under examination inclu

In [11]:
# 작업 셀 B — 셀 A 또는 pick() 이 출력한 set_label(...) 줄을 여기에 붙여넣고 고쳐서 실행한다.
# 예) set_label(1, "T2a", "confirmed", direction="unknown", quote="(원문 복사)", pid="15-p01234", note="Greater China 포함")


## 4. 라벨 검증

시트를 다 채운 뒤 실행한다. 모든 행이 `ok = True`가 될 때까지 시트를 고친다.
검증 항목: 허용 값, 인용 유무, **인용이 해당 문단에 원문 그대로 있는지**, not_found의 탐색 범위 기재.

In [12]:
labels = pd.read_csv(LABEL_PATH, encoding="utf-8-sig", keep_default_na=False)
for cik in labels["cik"].astype(int).unique():      # 추가 쌍으로 들어온 발행사 문서도 준비
    if cik not in docs:
        extract(sample[sample["cik"] == cik].iloc[0])
check = fl.validate_labels(labels, docs)
display(check[~check["ok"]] if (~check["ok"]).any() else "모든 행 통과")
print(labels["label"].value_counts().to_dict())

,cik,company,theme,label,ok,problems
0,1045810,NVIDIA CORP,T1,,False,label 값 '' 허용 안 됨
1,1045810,NVIDIA CORP,T2a,,False,label 값 '' 허용 안 됨
2,1045810,NVIDIA CORP,T2b,,False,label 값 '' 허용 안 됨
3,320193,Apple Inc.,T1,,False,label 값 '' 허용 안 됨
4,320193,Apple Inc.,T2a,,False,label 값 '' 허용 안 됨
5,320193,Apple Inc.,T2b,,False,label 값 '' 허용 안 됨
6,1652044,Alphabet Inc.,T1,,False,label 값 '' 허용 안 됨
7,1652044,Alphabet Inc.,T2a,,False,label 값 '' 허용 안 됨
8,1652044,Alphabet Inc.,T2b,,False,label 값 '' 허용 안 됨
9,789019,MICROSOFT CORP,T1,,False,label 값 '' 허용 안 됨


{'': 15}


### confirmed가 아닌 라벨이 4건 미만일 때 (프로토콜 7.1)

아래 셀을 한 번 실행할 때마다 다음 순서의 쌍(S1 rank 6의 T1 → T2a → T2b → rank 7 …)이 **한 개** 추가된다.
추가된 행을 라벨링한 뒤 다시 4절 검증을 실행하고, 4건 이상이 될 때까지 반복한다.

In [13]:
labels = pd.read_csv(LABEL_PATH, encoding="utf-8-sig", keep_default_na=False)
non_confirmed = int((labels["label"].isin(["possibly_related", "not_found"])).sum())
unlabeled = int((labels["label"] == "").sum())
print(f"confirmed가 아닌 라벨 {non_confirmed}건, 미작성 {unlabeled}건")

if unlabeled:
    print("→ 먼저 모든 행을 라벨링한다.")
elif non_confirmed >= 4:
    print("→ 조건 충족. 추가 쌍 필요 없음.")
else:
    done = set(zip(labels["rank"].astype(int), labels["theme"]))
    nxt = next((r, t) for r in sample["rank"] for t in ("T1", "T2a", "T2b") if (r, t) not in done)
    row = sample[sample["rank"] == nxt[0]].iloc[0]
    new = pd.DataFrame([{"rank": nxt[0], "cik": row.cik, "company": row.edgar_name,
                         "theme": nxt[1], "added_by_rule": "yes"}]).reindex(columns=fl.LABEL_COLUMNS).fillna("")
    pd.concat([labels, new]).to_csv(LABEL_PATH, index=False, encoding="utf-8-sig")
    if int(row.cik) not in docs:
        extract(row)
    print(f"→ 추가: rank {nxt[0]} {row.edgar_name} / {nxt[1]} — 라벨링 후 4절 검증을 다시 실행")

confirmed가 아닌 라벨 0건, 미작성 15건
→ 먼저 모든 행을 라벨링한다.


## 5. 커밋

검증을 모두 통과하면 **LLM 추출을 실행하기 전에** 라벨 시트를 커밋한다.

```bash
git add data/labeling/labels_P2.csv
git commit -m "data: human gold labels for P2 (before LLM extraction)"
git push
```

`experiment_log.md`에 라벨링 완료 일시, 라벨 분포, 추가 쌍 여부를 한 줄로 남긴다.